In [52]:
import torch
import torchvision
from torch import nn
from torchvision import transforms
from torchinfo import summary
from torchvision import models
from Going_Modular.Data_Setup import create_dataloaders
from Going_Modular.Train import train
from Going_Modular.Test import testing_pipeline
from Going_Modular.Summary_Writer import create_writer

In [2]:
weights=models.EfficientNet_B0_Weights.DEFAULT
weights

EfficientNet_B0_Weights.IMAGENET1K_V1

In [3]:
transform=weights.transforms()
transform

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [4]:
train_dir="pizza_steak_sushi/train"
test_dir="pizza_steak_sushi/test"

In [5]:
train_dataloader,test_dataloader,class_names=create_dataloaders(train_dir,test_dir,transform,transform,8)
train_dataloader,test_dataloader,class_names

(<torch.utils.data.dataloader.DataLoader at 0x220727123c0>,
 ['pizza', 'steak', 'sushi'])

In [6]:
model=models.efficientnet_b0(weights=weights)
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

In [7]:
for param in model.features.parameters():
    param.requires_grad=False

In [8]:
model.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

In [9]:
model.classifier=nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, out_features=len(class_names))
)

In [10]:
model.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=3, bias=True)
)

In [11]:
summary(model,input_size=(32, 3, 224, 224))

Layer (type:depth-idx)                                  Output Shape              Param #
EfficientNet                                            [32, 3]                   --
├─Sequential: 1-1                                       [32, 1280, 7, 7]          --
│    └─Conv2dNormActivation: 2-1                        [32, 32, 112, 112]        --
│    │    └─Conv2d: 3-1                                 [32, 32, 112, 112]        (864)
│    │    └─BatchNorm2d: 3-2                            [32, 32, 112, 112]        (64)
│    │    └─SiLU: 3-3                                   [32, 32, 112, 112]        --
│    └─Sequential: 2-2                                  [32, 16, 112, 112]        --
│    │    └─MBConv: 3-4                                 [32, 16, 112, 112]        (1,448)
│    └─Sequential: 2-3                                  [32, 24, 56, 56]          --
│    │    └─MBConv: 3-5                                 [32, 24, 56, 56]          (6,004)
│    │    └─MBConv: 3-6                      

In [12]:
loss_func=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001,weight_decay=0.0001)

In [13]:
from torch.utils.tensorboard import SummaryWriter

In [14]:
writer=SummaryWriter()

In [15]:
results=train(model,train_dataloader,test_dataloader,optimizer,loss_func,5,writer)

  0%|          | 0/5 [00:00<?, ?it/s]

-----Epoch 1-----
Training loss is 0.979274527779941
Training accuracy is 0.5474137931034483
-----Epoch 2-----
Training loss is 0.6897855061909248
Training accuracy is 0.8189655172413793
-----Epoch 3-----
Training loss is 0.5950175215458048
Training accuracy is 0.8232758620689655
-----Epoch 4-----
Training loss is 0.5047405608769121
Training accuracy is 0.8146551724137931
-----Epoch 5-----
Training loss is 0.4952754558160387
Training accuracy is 0.8189655172413793
Time taken for training is 73.4074760999938.


In [16]:
results

{'training_loss': [0.979274527779941,
  0.6897855061909248,
  0.5950175215458048,
  0.5047405608769121,
  0.4952754558160387],
 'training_accuracy': [0.5474137931034483,
  0.8189655172413793,
  0.8232758620689655,
  0.8146551724137931,
  0.8189655172413793],
 'test_loss': [0.840087479352951,
  0.6100473940372467,
  0.48677049577236176,
  0.441521979868412,
  0.4475307554006577],
 'test_accuracy': [0.6333333333333333, 0.8625, 0.85, 0.8875, 0.8625]}

In [21]:
%load_ext tensorboard
%tensorboard --logdir runs --port 6007

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [23]:
example_writer=create_writer("data_10_percent","effnetb0","5_epochs")

Created SummaryWriter at: runs\2026-08-28\data_10_percent\effnetb0\5_epochs


In [24]:
train_dir_10="pizza_steak_sushi/train"
train_dir_20="pizza_steak_sushi_20_percent/train"
test_dir="pizza_steak_sushi/test"

In [31]:
weights_b0=models.EfficientNet_B0_Weights.DEFAULT
transform_b0=weights_b0.transforms()
transform_b0

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [32]:
weights_b2=models.EfficientNet_B2_Weights.DEFAULT
transform_b2=weights_b2.transforms()
transform_b2

ImageClassification(
    crop_size=[288]
    resize_size=[288]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [33]:
train_dataloader_10_b0,test_dataloader_b0,class_names=create_dataloaders(train_dir_10,test_dir,transform_b0,transform_b0,8)
train_dataloader_20_b0,test_dataloader_b0,class_names=create_dataloaders(train_dir_20,test_dir,transform_b0,transform_b0,8)

In [34]:
len(train_dataloader_10),len(train_dataloader_20),len(test_dataloader),class_names

(29, 57, 10, ['pizza', 'steak', 'sushi'])

In [53]:
out_features=len(class_names)

In [54]:
def create_effnetb0():
    weights=models.EfficientNet_B0_Weights.DEFAULT
    model=models.efficientnet_b0(weights=weights)

    for param in model.features.parameters():
        param.requires_grad=False

    model.classifier=nn.Sequential(
        nn.Dropout(p=0.2,inplace=True),
        nn.Linear(in_features=1280,out_features=out_features)
    )

    model.name="effnetb0"
    print(f"Created new {model.name} model.")
    return model

In [55]:
def create_effnetb2():
    weights=models.EfficientNet_B2_Weights.DEFAULT
    model=models.efficientnet_b2(weights=weights)

    for param in model.features.parameters():
        param.requires_grad=False

    model.classifier=nn.Sequential(
        nn.Dropout(p=0.3,inplace=True),
        nn.Linear(in_features=1408,out_features=out_features)
    )

    model.name="effnetb2"
    print(f"Created new {model.name} model.")
    return model

In [56]:
effnetb0=create_effnetb0()
effnetb2=create_effnetb2()

Created new effnetb0 model.
Created new effnetb2 model.


In [57]:
epochs=[5,10]
model_names=["effnetb0","effnetb2"]
train_dataloaders={
    "data_10_percent":train_dataloader_10_b0,
    "data_20_percent":train_dataloader_20_b0
}

In [58]:
%%time
torch.manual_seed(18)
experiment_number=0
for epoch in epochs:
    for model_name in model_names:
        for dataloader_name,dataloader in train_dataloaders.items():
            experiment_number+=1
            print(f"Experiment number:{experiment_number}")
            print(f"Model name:{model_name}")
            print(f"Number of epochs:{epoch}")
            print(f"Dataloader:{dataloader_name}")

            if model_name=="effnetb0":
                model=create_effnetb0()
            else:
                model=create_effnetb2()

            loss_func=nn.CrossEntropyLoss()
            optimizer=torch.optim.Adam(model.parameters(),lr=0.001,weight_decay=0.0001)

            writer=create_writer(dataloader_name,model_name,extra=f"{epoch}_epochs")

            train(model,dataloader,test_dataloader,optimizer,loss_func,epoch,writer)

            print("-"*50)

Experiment number:1
Model name:effnetb0
Number of epochs:5
Dataloader:data_10_percent
Created new effnetb0 model.
Created SummaryWriter at: runs\2026-08-29\data_10_percent\effnetb0\5_epochs


  0%|          | 0/5 [00:00<?, ?it/s]

-----Epoch 1-----
Training loss is 1.0001239838271305
Training accuracy is 0.5086206896551724
-----Epoch 2-----
Training loss is 0.7552205611919535
Training accuracy is 0.728448275862069
-----Epoch 3-----
Training loss is 0.5845782571825487
Training accuracy is 0.8275862068965517
-----Epoch 4-----
Training loss is 0.5221602536481003
Training accuracy is 0.8146551724137931
-----Epoch 5-----
Training loss is 0.45773746484312516
Training accuracy is 0.8663793103448276
Time taken for training is 75.96593320000102.
--------------------------------------------------
Experiment number:2
Model name:effnetb0
Number of epochs:5
Dataloader:data_20_percent
Created new effnetb0 model.
Created SummaryWriter at: runs\2026-08-29\data_20_percent\effnetb0\5_epochs


  0%|          | 0/5 [00:00<?, ?it/s]

-----Epoch 1-----
Training loss is 0.8668286831755387
Training accuracy is 0.6381578947368421
-----Epoch 2-----
Training loss is 0.6162009866614091
Training accuracy is 0.8026315789473685
-----Epoch 3-----
Training loss is 0.51116430158155
Training accuracy is 0.8267543859649122
-----Epoch 4-----
Training loss is 0.45282279974535894
Training accuracy is 0.8508771929824561
-----Epoch 5-----
Training loss is 0.41456950573544754
Training accuracy is 0.8618421052631579
Time taken for training is 134.21233260000008.
--------------------------------------------------
Experiment number:3
Model name:effnetb2
Number of epochs:5
Dataloader:data_10_percent
Created new effnetb2 model.
Created SummaryWriter at: runs\2026-08-29\data_10_percent\effnetb2\5_epochs


...

-----Epoch 1-----
Training loss is 0.8936911758623625
Training accuracy is 0.6293859649122807
-----Epoch 2-----
Training loss is 0.5926072079884378
Training accuracy is 0.8026315789473685
-----Epoch 3-----
Training loss is 0.4895264600452624
Training accuracy is 0.831140350877193
-----Epoch 4-----
Training loss is 0.48751882476764813
Training accuracy is 0.8267543859649122
-----Epoch 5-----
Training loss is 0.414844145377477
Training accuracy is 0.8706140350877193
-----Epoch 6-----
Training loss is 0.36323558186229904
Training accuracy is 0.8859649122807017
-----Epoch 7-----
Training loss is 0.3486248241704807
Training accuracy is 0.8881578947368421
-----Epoch 8-----
Training loss is 0.3838899105525853
Training accuracy is 0.8552631578947368
-----Epoch 9-----
Training loss is 0.3819318249037391
Training accuracy is 0.8552631578947368
-----Epoch 10-----
Training loss is 0.3434697625537713
Training accuracy is 0.8771929824561403
Time taken for training is 257.2530920999998.
-------------

  0%|          | 0/10 [00:00<?, ?it/s]

-----Epoch 1-----
Training loss is 0.9763484103926297
Training accuracy is 0.5775862068965517
-----Epoch 2-----
Training loss is 0.7354804739869875
Training accuracy is 0.771551724137931
-----Epoch 3-----
Training loss is 0.6472763378044655
Training accuracy is 0.8017241379310345
-----Epoch 4-----
Training loss is 0.5475597648785032
Training accuracy is 0.8448275862068966
-----Epoch 5-----
Training loss is 0.4735119378772275
Training accuracy is 0.8620689655172413
-----Epoch 6-----
Training loss is 0.44575350808686226
Training accuracy is 0.8362068965517241
-----Epoch 7-----
Training loss is 0.4187108401594491
Training accuracy is 0.8620689655172413
-----Epoch 8-----
Training loss is 0.4114942602042494
Training accuracy is 0.853448275862069
-----Epoch 9-----
Training loss is 0.4111005350433547
Training accuracy is 0.8620689655172413
-----Epoch 10-----
Training loss is 0.3505109104103056
Training accuracy is 0.8922413793103449
Time taken for training is 203.97050459997263.
-------------

  0%|          | 0/10 [00:00<?, ?it/s]

-----Epoch 1-----
Training loss is 0.8762886670597813
Training accuracy is 0.6622807017543859
-----Epoch 2-----
Training loss is 0.6041682048847801
Training accuracy is 0.8333333333333334
-----Epoch 3-----
Training loss is 0.5297981469254744
Training accuracy is 0.8464912280701754
-----Epoch 4-----
Training loss is 0.4405655547192222
Training accuracy is 0.8706140350877193
-----Epoch 5-----
Training loss is 0.4364390890849264
Training accuracy is 0.8508771929824561
-----Epoch 6-----
Training loss is 0.43066417177518207
Training accuracy is 0.8552631578947368
-----Epoch 7-----
Training loss is 0.3506084285807191
Training accuracy is 0.881578947368421
-----Epoch 8-----
Training loss is 0.3615457367217332
Training accuracy is 0.8771929824561403
-----Epoch 9-----
Training loss is 0.3799395484098217
Training accuracy is 0.8706140350877193
-----Epoch 10-----
Training loss is 0.33072368669928165
Training accuracy is 0.8947368421052632
Time taken for training is 383.348192400008.
-------------

In [59]:
%load_ext tensorboard
%tensorboard --logdir runs --port 6007

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 21564), started 3:15:58 ago. (Use '!kill 21564' to kill it.)